# FUZZY STRESS LEVEL CLASSIFIER
## Sistem Pendukung Keputusan: Prediksi Tingkat Stres Mahasiswa
**Menggunakan Algoritma Logika Fuzzy Mamdani**
Notebook ini berisi implementasi komputasi Sistem Pendukung Keputusan (SPK) untuk mengevaluasi dan memprediksi tingkat stres mahasiswa. Sistem dibangun menggunakan **Sistem Inferensi Fuzzy (Metode Mamdani)** yang mengeksekusi 81 basis aturan (*Rule Base*) menggunakan pustaka `scikit-fuzzy`.

**Variabel Input (Skala 0 - 5):**
1. **Beban Akademik** (*Study Load*)
2. **Kualitas Tidur** (*Sleep Quality*)
3. **Keaktifan Organisasi** (*Extracurricular Activities*) — *Modifikasi Konteks Lokal*
4. **Kecemasan Karir** (*Future Career Concerns*)

**Variabel Output (Skala 0 - 2):**
* **Tingkat Stres** (*Stress Level*: Rendah, Sedang, Tinggi)

**Validasi & Evaluasi:**
Sistem ini dievaluasi secara kuantitatif dengan membandingkan nilai hasil *defuzzifikasi Centroid of Area (CoA)* melawan data observasi riil dari dataset publik Kaggle (*Student Stress Factors: A Comprehensive Analysis*).

Deklarasi Variabel Input dan Output

In [1]:
!pip install numpy scikit-fuzzy networkx scipy packaging pandas scikit-learn

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import itertools
import pandas as pd
from sklearn.metrics import accuracy_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 377.4 kB/s  0:00:25m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 844.2 kB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]learn]


Membership Functions

In [2]:
#Input: Skala 0.0 sampai 5.0
study_load = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'study_load')
sleep_quality = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'sleep_quality')
extracurricular = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'extracurricular')
career_concern = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'career_concern')

# Output: Skala 0.0 sampai 2.0
stress_level = ctrl.Consequent(np.arange(0, 2.1, 0.1), 'stress_level')

study_load.automf(names=['ringan', 'sedang', 'berat'])
sleep_quality.automf(names=['buruk', 'cukup', 'baik'])
extracurricular.automf(names=['pasif', 'aktif', 'sangat_aktif'])
career_concern.automf(names=['rendah', 'sedang', 'tinggi'])

stress_level.automf(names=['rendah', 'sedang', 'tinggi'])

Rule Base

In [3]:
terms_study = ['ringan', 'sedang', 'berat']         # Skor: 0, 1, 2
terms_sleep = ['baik', 'cukup', 'buruk']            # Skor: 0, 1, 2 
terms_extra = ['pasif', 'aktif', 'sangat_aktif']    # Skor: 0, 1, 2
terms_career = ['rendah', 'sedang', 'tinggi']       # Skor: 0, 1, 2

daftar_aturan = []

# Loop ini akan memutar semua 81 kemungkinan kombinasi (3x3x3x3)
for st, sl, ex, ca in itertools.product(range(3), range(3), range(3), range(3)):
    # Hitung total skor bobot keparahan (Max skor = 8, Min skor = 0)
    total_skor = st + sl + ex + ca
    
    # Logika Pakar (Expert Logic) untuk menentukan tingkat stres
    if total_skor <= 2:
        out_term = 'rendah'
    elif total_skor <= 5:
        out_term = 'sedang'
    else:
        out_term = 'tinggi'
        
    # Pembuatan Rule secara dinamis
    rule = ctrl.Rule(
        study_load[terms_study[st]] & 
        sleep_quality[terms_sleep[sl]] & 
        extracurricular[terms_extra[ex]] & 
        career_concern[terms_career[ca]], 
        stress_level[out_term]
    )
    daftar_aturan.append(rule)

Kontrol Sistem & Simulasi

In [4]:
stress_ctrl = ctrl.ControlSystem(daftar_aturan)
stress_simulasi = ctrl.ControlSystemSimulation(stress_ctrl)

Fungsi Pengujian Kasus

In [5]:
def uji_kasus(nama_kasus, load, sleep, extra, career):
    print(f"\n--- Menguji {nama_kasus} ---")
    print(f"Input -> Tugas: {load}, Tidur: {sleep}, Organisasi: {extra}, Karir: {career}")
    
    stress_simulasi.input['study_load'] = load
    stress_simulasi.input['sleep_quality'] = sleep
    stress_simulasi.input['extracurricular'] = extra
    stress_simulasi.input['career_concern'] = career
    
    # Eksekusi Mamdani & Centroid of Area
    stress_simulasi.compute()
    hasil = stress_simulasi.output['stress_level']
    
    print(f"Skor Defuzzifikasi (0-2) : {hasil:.2f}")
    
    if hasil >= 1.15:
        print("Kategori Sistem        : STRES TINGGI")
    elif hasil >= 0.85:
        print("Kategori Sistem        : STRES SEDANG")
    else:
        print("Kategori Sistem        : STRES RENDAH")
    print("-" * 35)


Uji 3 Kasus

In [6]:
print("=== SISTEM INFERENSI FUZZY MAMDANI: PREDIKSI STRES ===")
uji_kasus("Kasus 1 (Burnout Ekstrem)", load=4.5, sleep=1.0, extra=4.0, career=4.0)
uji_kasus("Kasus 2 (Mahasiswa Santai)", load=1.0, sleep=4.5, extra=1.0, career=1.0)
uji_kasus("Kasus 3 (Tekanan Menengah)", load=3.0, sleep=3.0, extra=3.0, career=3.0)

=== SISTEM INFERENSI FUZZY MAMDANI: PREDIKSI STRES ===

--- Menguji Kasus 1 (Burnout Ekstrem) ---
Input -> Tugas: 4.5, Tidur: 1.0, Organisasi: 4.0, Karir: 4.0
Skor Defuzzifikasi (0-2) : 1.18
Kategori Sistem        : STRES TINGGI
-----------------------------------

--- Menguji Kasus 2 (Mahasiswa Santai) ---
Input -> Tugas: 1.0, Tidur: 4.5, Organisasi: 1.0, Karir: 1.0
Skor Defuzzifikasi (0-2) : 0.82
Kategori Sistem        : STRES RENDAH
-----------------------------------

--- Menguji Kasus 3 (Tekanan Menengah) ---
Input -> Tugas: 3.0, Tidur: 3.0, Organisasi: 3.0, Karir: 3.0
Skor Defuzzifikasi (0-2) : 1.02
Kategori Sistem        : STRES SEDANG
-----------------------------------


Evaluasi Dataset

In [7]:
# Load file CSV
try:
    df = pd.read_csv('StressLevelDataset.csv')
    print("Dataset berhasil dimuat!")
    
    # Pilih 100 data acak untuk pengujian cepat
    sample_df = df.sample(100, random_state=42)
    
    label_asli = sample_df['stress_level'].values
    hasil_prediksi_fuzzy = []

    print("Sedang menghitung akurasi sistem...")

    for index, row in sample_df.iterrows():
        # Input data dari baris CSV ke input sistem Fuzzy
        stress_simulasi.input['study_load'] = row['study_load']
        stress_simulasi.input['sleep_quality'] = row['sleep_quality']
        stress_simulasi.input['extracurricular'] = row['extracurricular_activities']
        stress_simulasi.input['career_concern'] = row['future_career_concerns']
        
        # Hitung
        stress_simulasi.compute()
        out = stress_simulasi.output['stress_level']
        
        # Konversi skor desimal ke kategori 0, 1, atau 2 sesuai threshold tuning
        if out >= 1.15:
            prediksi = 2 # Tinggi
        elif out >= 0.85:
            prediksi = 1 # Sedang
        else:
            prediksi = 0 # Rendah
            
        hasil_prediksi_fuzzy.append(prediksi)

    # Hitung Skor Akurasi
    akurasi = accuracy_score(label_asli, hasil_prediksi_fuzzy)
    print(f"\n=== HASIL EVALUASI KINERJA ===")
    print(f"Akurasi Sistem terhadap Dataset Kaggle: {akurasi * 100:.2f}%")

except FileNotFoundError:
    print("Error: File 'StressLevelDataset.csv' tidak ditemukan. Pastikan file ada di folder yang sama dengan notebook ini.")

Dataset berhasil dimuat!
Sedang menghitung akurasi sistem...

=== HASIL EVALUASI KINERJA ===
Akurasi Sistem terhadap Dataset Kaggle: 83.00%
